# Training results

Reads `outputs/train_history.csv` and plots the training loss and validation MAE across epochs. The retained checkpoint (lowest validation MAE) is marked in red, and the epochs that set a new best are listed.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import config

HIST = (ROOT / config.OUTPUT_DIR / "train_history.csv").resolve()
h = pd.read_csv(HIST)

# best_mae is filled only on epochs that set a new best (see scripts/train.py)
best_rows = h[h["best_mae"].notna()].copy()
best_epoch = int(h.loc[h["val_mae"].idxmin(), "epoch"])
best_mae = float(h["val_mae"].min())

print(f"epochs run: {len(h)}  |  best epoch: {best_epoch}  |  best val MAE: {best_mae:.4f}")
print("\nepochs that set a new best:")
print(best_rows[["epoch", "best_mae"]].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(h["epoch"], h["train_loss"], marker="o", ms=3, lw=1.5,
        color="#5a4fcf", label="Training loss (masked L1)")
ax.plot(h["epoch"], h["val_mae"], marker="s", ms=3, lw=1.5,
        color="#e07b39", label="Validation MAE")

# mark the retained checkpoint (lowest validation MAE)
ax.scatter([best_epoch], [best_mae], color="red", s=100, marker="*", zorder=5)

ax.set_xlabel("Epoch")
ax.set_ylabel("Masked error (normalized [0, 1])")
ax.set_title("Training loss and validation MAE across epochs")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)

# best-MAE progression, right under the legend (final one in red)
ax.text(0.98, 0.80, "Best validation MAE", transform=ax.transAxes,
        va="top", ha="right", fontsize=9, fontweight="bold")
for i, (_, r) in enumerate(best_rows.iterrows()):
    ep = int(r["epoch"]); mae = float(r["best_mae"])
    final = ep == best_epoch
    ax.text(0.98, 0.73 - i * 0.055, f"epoch {ep:>2d} = {mae:.4f}",
            transform=ax.transAxes, va="top", ha="right", family="monospace",
            fontsize=9, color="red" if final else "#333333",
            fontweight="bold" if final else "normal")

plt.tight_layout()
plt.show()